# 埃尔米特多项式离线计算器（非线性回归的线性拟合）

**第一步：选择约定（两套权函数与正交基）**

| 约定 | 权函数 | 正交基 | 递推关系 |
|---|---|---|---|
| 物理学家版 | $f(x)=e^{-x^2}$ | 物理学家版 $\{H_n\}$ | $H_0=1,\ H_1=2x,\ H_{n+1}=2xH_n-2nH_{n-1}$ |
| 概率学家版 | $\varphi(x)=\dfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$（标准正态密度） | 概率学家版 $\{He_n\}$ | $He_0=1,\ He_1=x,\ He_{n+1}=xHe_n-nHe_{n-1}$ |

两套基的换算关系：$He_n(x)=2^{-n/2}H_n\!\left(\dfrac{x}{\sqrt2}\right)$，还原的是同一个函数。

**标准称呼对照**

- 内积：物理学家版 $\langle p,q\rangle_f=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx$；
  概率学家版 $\langle p,q\rangle_\varphi=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)\varphi(x)\,dx$
- 你说的“函数向量的系数基”，标准名称是 **埃尔米特展开系数（广义傅里叶系数）**：
  $\alpha_n=\dfrac{\langle h,P_n\rangle_w}{\langle P_n,P_n\rangle_w}$，于是 $h(x)=\displaystyle\sum_{n=0}^{N}\alpha_n P_n(x)$
  （$P_n$ 按约定取 $H_n$ 或 $He_n$）
- Rodrigues 记号：物理学家版 $\dfrac{f^{(n)}}{f}=(-1)^n H_n$；
  概率学家版 $\dfrac{\varphi^{(n)}}{\varphi}=(-1)^n He_n$，该基下的系数向量本工具同时给出

**两种模式（对应两个 Tab）**

| | 输入 | 对应 Cell | 方法 |
|---|---|---|---|
| 模式一 | 已知 $h$：输入阶数 $d$ 与各阶系数 $a_k$ | Cell 4 | 精确有理运算，输出分数形式的 $\alpha_n$ 与系数向量 |
| 模式二 | 未知 $h$：输入 $m$ 个点坐标 $(x_i,y_i)$ 与最高阶 $N$ | Cell 5 | 笔记公式 III 梯形数值积分，输出完整多项式方程 |

**模式二停机指标 $\Delta_n$ 的计算原理**

- 部分和：$P_n(x)=\displaystyle\sum_{j=0}^{n}\alpha_jB_j(x)$（$B_j$ 按约定取 $H_j$ 或 $He_j$）；真函数 $\varphi=\lim_{n\to\infty}P_n$，工具当前输出的拟合多项式 $F=P_N$。
- 定义（相邻两阶部分和的加权范数相对增量，$n\ge1$，$\Delta_0$ 无定义）：

$$\Delta_n=\frac{\|P_n-P_{n-1}\|_w}{\|P_n\|_w}
=\sqrt{\frac{\displaystyle\int (P_n-P_{n-1})^2w(x)\,dx}{\displaystyle\int P_n^2(x)w(x)\,dx}}
=\frac{\|\alpha_nB_n\|_w}{\|P_n\|_w},\qquad P_n-P_{n-1}=\alpha_nB_n$$

- 平方差展开后，积分号里会出现老师红笔板书中“$P_n$ 与 $P_{n-1}$ 相乘、不平方”的**乘积交叉项** $P_nP_{n-1}$：

$$\int (P_n-P_{n-1})^2w\,dx
=\int P_n^2w\,dx-2\int P_n(x)P_{n-1}(x)w(x)\,dx+\int P_{n-1}^2w\,dx$$

- 板书对照（重要）：蓝笔范数定义与上面的差平方积分是标准口径；老师红笔“i.e.”在演算区简写过乘积根号形式 $\sqrt{\int P_nP_{n-1}w\,dx\big/\int P_n^2w\,dx}$。单独的 $\int P_nP_{n-1}w=\langle P_n,P_{n-1}\rangle_w$ 只是展开式里的交叉内积项，**不等于**分子 $\|P_n-P_{n-1}\|_w^2$；精确正交时它等于 $\|P_{n-1}\|_w^2$，该字面比值会随 $n$ 趋近 1，不能度量“相邻阶变化量”。默认 6 点作业数据（物理版、节点梯形）按乘积字面式复算约为 $0.722/0.971/0.998/0.997/1.001$，与白板左侧数值列 $0.506050/0.123712/0.092488/0.042527/0.021085$ 不吻合；本工具第三列采用与白板数值逐一吻合的**差平方范数口径**，建议课上再与老师确认红笔原式是展开式还是漏写了 $(P_n-P_{n-1})^2$。
- 离散实现：分子、分母两个加权平方积分都在**数据节点上用梯形法**求和（不是全实轴解析范数）；Cell 5 结果区给出连续积分形式、完整离散公式与逐行数值。
- 停机规则：课堂建议阈值 $\Delta_N<0.05$ 时可停止加阶、取 $\varphi\approx P_N$；注意 $\Delta_n$ 是开根号后的**范数比**，$\Delta_n^2$ 才是平方能量比。红色欠定（点数少于参数数）时停机信号仅供演示，需结合 SSE、残差表与三级点数警告一起判断。

所有公式均以 LaTeX 规范排版，分数为上下堆叠形式；模式二的分数为小数的近似分数（分母上限 10000）。

In [1]:
# ===== Cell 2｜依赖导入与全局配置（无需修改，直接运行）=====
import re
import numpy as np
import sympy as sp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Latex, Markdown, clear_output

x = sp.symbols('x')
%matplotlib inline

In [2]:
# ===== Cell 3｜核心数学函数（双约定参数化，无需修改）=====

# ---------- 约定配置：a 为递推尺度（物理学家版 2，概率学家版 1）----------
CONV = {
    'phys': {
        'a': 2,
        'weight_num': lambda xv: np.exp(-xv ** 2),
        'bname': 'H', 'wname': 'f',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx',
        'weight_tex': r'e^{-x^2}',
    },
    'prob': {
        'a': 1,
        'weight_num': lambda xv: np.exp(-xv ** 2 / 2.0),
        'bname': 'He', 'wname': r'\varphi',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)\frac{1}{\sqrt{2\pi}}e^{-x^2/2}\,dx',
        'weight_tex': r'\frac{1}{\sqrt{2\pi}}e^{-x^2/2}',
    },
}
CONV_NAME = 'phys'

# ---------- 精确模式 ----------
def parse_rational(s):
    """解析整数、小数、分数（如 7/2、-1/4），返回精确有理数 sp.Rational"""
    s = str(s).strip().replace(' ', '')
    if '/' in s:
        p, q = s.split('/')
        return sp.Rational(int(p), int(q))
    return sp.Rational(s)

def exact_basis_poly(n, conv_name):
    """递推构造第 n 阶基多项式（精确）：P_{k+1} = a x P_k - a k P_{k-1}"""
    a = sp.Integer(CONV[conv_name]['a'])
    if n == 0:
        return sp.Integer(1)
    p_prev, p_cur = sp.Integer(1), a * x
    for k in range(1, n):
        p_prev, p_cur = p_cur, sp.expand(a * x * p_cur - a * k * p_prev)
    return p_cur

def basis_matrix(d, conv_name):
    """M[k,n] = P_n 中 x^k 的系数；普通系数 a 与展开系数 alpha 满足 a = M alpha"""
    M = sp.zeros(d + 1, d + 1)
    for n in range(d + 1):
        for monom, c in sp.Poly(exact_basis_poly(n, conv_name), x).terms():
            M[monom[0], n] = c
    return M

def exact_expansion(coeffs, conv_name):
    """输入普通多项式升幂系数 [a0,...,ad]，返回 alpha（P_n 基）、beta（Rodrigues 基）、h 表达式"""
    d = len(coeffs) - 1
    alpha = [sp.simplify(v) for v in basis_matrix(d, conv_name).solve(sp.Matrix(coeffs))]
    beta = [((-1) ** n) * v for n, v in enumerate(alpha)]
    h_expr = sp.expand(sum(coeffs[k] * x ** k for k in range(d + 1)))
    return alpha, beta, h_expr

# ---------- 数值模式 ----------
def basis_coeff_arrays(N, conv_name):
    """递推构造 P_0...P_N 的升幂系数数组（float）"""
    a = float(CONV[conv_name]['a'])
    P = [np.array([1.0]), np.array([0.0, a])]
    for n in range(1, N):
        shifted = np.concatenate(([0.0], a * P[n]))            # a x P_n
        prev_pad = np.zeros(len(shifted))
        prev_pad[:len(P[n - 1])] = P[n - 1]
        P.append(shifted - a * n * prev_pad)
    return P[:N + 1]

def basis_values(xv, N, conv_name):
    """P_n 在节点 xv 上的取值，返回 (N+1) x m 数组"""
    arrays = basis_coeff_arrays(N, conv_name)
    return np.vstack([np.polyval(p[::-1], xv) for p in arrays])

def trapezoid_alpha(xv, yv, N, conv_name):
    """笔记公式 III：在节点区间 [x_1,x_m] 上用梯形法数值计算 alpha_0...alpha_N"""
    Pv = basis_values(xv, N, conv_name)
    weight = CONV[conv_name]['weight_num'](xv)
    dx = np.diff(xv)
    alpha = []
    for n in range(N + 1):
        g_num = yv * Pv[n] * weight
        g_den = Pv[n] ** 2 * weight
        num = np.sum(dx / 2.0 * (g_num[1:] + g_num[:-1]))
        den = np.sum(dx / 2.0 * (g_den[1:] + g_den[:-1]))
        alpha.append(num / den)
    return np.array(alpha, dtype=float), Pv

def trapezoid_delta(xv, Pv, alpha, conv_name):
    """课堂 Truncation Rule（老师白板口径）：
    Delta_n = ||P_n-P_{n-1}||_w / ||P_n||_w，两个范数都在数据节点上用梯形法计算。
    注意左侧 P_n 是部分和；P_n-P_{n-1} = alpha_n*B_n(x)，其中 B_n=H_n/He_n 是第 n 阶基多项式
    （即入参数组 Pv[n]，不是部分和 P_n）。返回长度 N+1 数组，delta[0]=nan。"""
    weight = CONV[conv_name]['weight_num'](xv)
    dx = np.diff(xv)
    def _trap(g):
        return float(np.sum(dx / 2.0 * (g[1:] + g[:-1])))
    N = len(alpha) - 1
    out = np.full(N + 1, np.nan)
    Pn_nodes = np.zeros_like(xv, dtype=float)
    for n in range(N + 1):
        Pn_nodes = Pn_nodes + alpha[n] * Pv[n]
        if n >= 1:
            dP = alpha[n] * Pv[n]
            num = _trap(dP ** 2 * weight)
            den = _trap(Pn_nodes ** 2 * weight)
            out[n] = float(np.sqrt(num / den)) if den > 0 else np.nan
    return out

def expand_numeric(alpha, conv_name):
    """把 sum alpha_n P_n 合并为普通多项式升幂系数"""
    arrays = basis_coeff_arrays(len(alpha) - 1, conv_name)
    c = np.zeros(max(len(p) for p in arrays))
    for n, a in enumerate(alpha):
        c[:len(arrays[n])] += a * arrays[n]
    return c

def approx_frac(v, max_den=10000):
    """小数转近似分数，用于规范 LaTeX 显示"""
    return sp.Rational(float(v)).limit_denominator(max_den)

def _in_fixed_range(v):
    """常规量级判定：此范围内才显示定点小数与近似分数"""
    av = abs(float(v))
    return v != 0 and av < 1e5 and av >= 1e-4

def _sci_tex(v):
    """科学计数法的 LaTeX 形式，如 2.711\\times 10^{4}"""
    mant, exp = ('%.3e' % float(v)).split('e')
    return r'%s\times 10^{%d}' % (mant, int(exp))

def fmt_num(v, fixed=6, latex=False):
    """常规量级用定点小数；过大（≥1e5）或过小（<1e-4）用科学计数法，3 位有效尾数"""
    if float(v) == 0:
        return '0'
    if not _in_fixed_range(v):
        return _sci_tex(v) if latex else '%.3e' % float(v)
    return ('%.' + str(fixed) + 'f') % float(v)

def _coef_parts(v, approx=True):
    """系数拆为 (负号, 主体 LaTeX)；常规量级用近似分数，超范围用科学计数法；零返回 None"""
    if float(v) == 0:
        return None
    if _in_fixed_range(v):
        c = approx_frac(v) if approx else sp.simplify(v)
        return _split_sign(c)
    return float(v) < 0, fmt_num(abs(float(v)), latex=True)

def L(s):
    """把裸 LaTeX 包进 gathered 环境，保证前端按公式渲染而非显示源码"""
    return Latex(r'\begin{gathered} ' + s + r' \end{gathered}')

def _split_sign(c):
    """拆出系数的正负号与主体 LaTeX，避免替换字符串产生双空格"""
    s = sp.latex(c)
    if s.startswith('-'):
        return True, s[1:].lstrip()
    return False, s

def _join_signed(signed_terms):
    """signed_terms 为 (是否负号, 主体 LaTeX) 列表，按规范正负号拼接"""
    if not signed_terms:
        return '0'
    neg, body = signed_terms[0]
    s = '-' + body if neg else body
    for neg, body in signed_terms[1:]:
        s += (' - ' if neg else ' + ') + body
    return s

def latex_basis_form(alpha, conv_name, approx=True):
    """alpha 序列转 sum alpha_n P_n 的 LaTeX"""
    bn = CONV[conv_name]['bname']
    signed = []
    for n, v in enumerate(alpha):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        signed.append((neg, body + r'\,%s_{%d}(x)' % (bn, n)))
    return _join_signed(signed)

def latex_standard_poly(c_asc, approx=True):
    """升幂系数转标准多项式 LaTeX"""
    signed = []
    for k, v in enumerate(c_asc):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        if k == 0:
            tail = ''
        elif k == 1:
            tail = r'\,x'
        else:
            tail = r'\,x^{%d}' % k
        signed.append((neg, body + tail))
    return _join_signed(signed)

def rodrigues_tex(conv_name):
    """Rodrigues 恒等式的 LaTeX"""
    if conv_name == 'phys':
        return r'\frac{f^{(n)}}{f}=(-1)^n H_n'
    return r'\frac{\varphi^{(n)}}{\varphi}=(-1)^n He_n'

def static_trap_tex(conv_name):
    """笔记公式 III 完整 LaTeX（按约定替换基符号与权函数）"""
    bn = CONV[conv_name]['bname']
    if conv_name == 'phys':
        w = lambda i: r'e^{-x_{%s}^2}' % i
    else:
        w = lambda i: r'\frac{1}{\sqrt{2\pi}}e^{-x_{%s}^2/2}' % i
    return (
        r'\alpha_n \approx \frac{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[y_{i+1}%s_n(x_{i+1})%s+y_i%s_n(x_i)%s\right]}{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[%s_n^2(x_{i+1})%s+%s_n^2(x_i)%s\right]}'
        % (bn, w('i+1'), bn, w('i'), bn, w('i+1'), bn, w('i')))

def static_delta_tex(conv_name):
    """停机指标 Δ_n 的节点梯形离散公式（在数据节点上做梯形求和）"""
    bn = CONV[conv_name]['bname']
    if conv_name == 'phys':
        w = lambda i: r'e^{-x_{%s}^2}' % i
    else:
        w = lambda i: r'\frac{1}{\sqrt{2\pi}}e^{-x_{%s}^2/2}' % i
    w1, w0 = w('i+1'), w('i')
    return (
        r'\Delta_n \approx \sqrt{\frac{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[(\alpha_n %s_n(x_{i+1}))^2%s+(\alpha_n %s_n(x_i))^2%s\right]}{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[P_n(x_{i+1})^2%s+P_n(x_i)^2%s\right]}}'
        % (bn, w1, bn, w0, w1, w0))

def order_warning(m, N):
    """三级阶数提示：返回 (level, text)，level 取 ok / caution / danger（p=N+1 为待估参数数）"""
    p = N + 1
    if m < p:
        return 'danger', ('待估参数 %d 个多于点数 %d，问题欠定、解不唯一，'
                          '以下结果仅作演示，不具统计意义' % (p, m))
    if m < 2 * p:
        return 'caution', ('当前 %d 个点估计 %d 个系数，每参数不足 2 个点，'
                           '系数可能不稳定，请谨慎引用' % (m, p))
    return 'ok', ''

_WARN_STYLE = {
    'caution': ('#fef3c7', '#f59e0b', '#92400e'),
    'danger': ('#fee2e2', '#ef4444', '#991b1b'),
}

def warning_banner(m, N):
    """生成黄/红提示横幅 Markdown；ok 级别返回空字符串"""
    level, text = order_warning(m, N)
    if level == 'ok':
        return ''
    bg, border, color = _WARN_STYLE[level]
    return Markdown(
        '<div style="background:%s;border:1px solid %s;color:%s;border-radius:10px;'
        'padding:10px 14px;margin:10px 0;font-size:13.5px">%s</div>'
        % (bg, border, color, text))

def stop_banner(delta_arr, N, warn_level='ok', threshold=0.05):
    """课堂 Truncation Rule 停机提示：当前最高阶 N 的 Delta_N<threshold 给绿色“可停止”；
    若同时是红色欠定情形则降级为琥珀色提醒；Delta_N>=threshold 给中性蓝提示。"""
    if (N < 1 or delta_arr is None or N >= len(delta_arr)
            or not np.isfinite(delta_arr[N])):
        return ''
    dN = float(delta_arr[N])
    first = next((n for n in range(1, N + 1)
                  if np.isfinite(delta_arr[n]) and delta_arr[n] < threshold), None)
    cross = ('' if first is None or first == N else
             ' 当前 1–%d 范围内 Δ_n 首次低于 %.2f 出现在 n=%d；稀疏节点下高阶可能反弹，'
             '逐阶增加时建议连续两阶低于阈值再停止。' % (N, threshold, first))
    if dN < threshold:
        if warn_level == 'danger':
            bg, bd, fg = '#fef3c7', '#f59e0b', '#92400e'
            head = ('Δ_%d = %.6f &lt; %.2f，形式上达到课堂停机阈值；但当前为红色欠定情形'
                    '（待估参数多于点数），该停机信号不稳定、结果仅作演示，建议先增加数据点。'
                    % (N, dN, threshold))
        else:
            bg, bd, fg = '#dcfce7', '#22c55e', '#166534'
            head = ('Δ_%d = %.6f &lt; %.2f：相邻两阶部分和在节点梯形加权范数下的相对变化已小于 5%%，'
                    '按课堂 Truncation Rule（stability of P_n）可在此阶停止，取 φ ≈ P_%d；'
                    '请同时参考 SSE、残差表与点数警告。' % (N, dN, threshold, N))
    else:
        bg, bd, fg = '#eff6ff', '#3b82f6', '#1e3a8a'
        head = ('Δ_%d = %.6f ≥ %.2f：第 %d 阶相对增量仍不小于 5%%，按稳定性准则可继续提高阶数观察；'
                '若同时出现黄/红点数警告、SSE 反弹或高阶振荡，应综合判断，不要盲目加阶。'
                % (N, dN, threshold, N))
    return Markdown(
        '<div style="background:%s;border:1px solid %s;color:%s;border-radius:10px;'
        'padding:10px 14px;margin:10px 0;font-size:13.5px;line-height:1.6">%s%s</div>'
        % (bg, bd, fg, head, cross))

In [3]:
# ===== Cell 4｜模式一：已知 h(x)，输入阶数 d 与各阶系数 a_k，运行本 Cell 出界面 =====
conv_select1 = widgets.ToggleButtons(
    options=[('物理学家版 H_n（权 f=e^{-x²}）', 'phys'),
             ('概率学家版 He_n（权 φ=标准正态）', 'prob')],
    value=CONV_NAME, description='约定选择', style={'description_width': 'initial'})

def _on_conv1(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select2' in globals() and conv_select2.value != change['new']:
        conv_select2.value = change['new']

conv_select1.observe(_on_conv1, names='value')

degree_box = widgets.BoundedIntText(value=2, min=0, max=12, description='阶数 d')
coeff_box = widgets.VBox()

def rebuild_coeff_inputs(_=None):
    coeff_box.children = [
        widgets.HBox([
            widgets.Label('a_%d（x^%d 项系数）：' % (k, k)),
            widgets.Text(value='0', layout=widgets.Layout(width='150px'))
        ]) for k in range(degree_box.value + 1)
    ]

degree_box.observe(rebuild_coeff_inputs, names='value')
rebuild_coeff_inputs()
mode1_out = widgets.Output(layout=widgets.Layout(
    max_height='560px', overflow='auto', border='1px solid #e5e7eb',
    padding='8px 10px', margin='8px 0'))

def fill_mode1(vals):
    """一键填充例题：vals 为升幂系数字符串列表"""
    degree_box.value = len(vals) - 1
    rebuild_coeff_inputs()
    for k, v in enumerate(vals):
        coeff_box.children[k].children[1].value = v

def run_mode1(_):
    with mode1_out:
        clear_output(wait=True)
        try:
            coeffs = [parse_rational(row.children[1].value) for row in coeff_box.children]
            if all(c == 0 for c in coeffs):
                raise ValueError('系数不能全为 0')
            conv = conv_select1.value
            cfg = CONV[conv]
            bn, wn = cfg['bname'], cfg['wname']
            alpha, beta, h_expr = exact_expansion(coeffs, conv)
            d = len(coeffs) - 1
            display(L(r'输入函数：\quad h(x) = ' + sp.latex(h_expr)))
            basis = r' \\ '.join(
                r'%s_{%d}(x) = %s' % (bn, n, sp.latex(exact_basis_poly(n, conv)))
                for n in range(d + 1))
            display(Latex(r'使用的正交基：$\begin{align*} %s \end{align*}$' % basis))
            display(L(
                r'\alpha_n = \frac{\langle h,%s_n\rangle_{%s}}{\langle %s_n,%s_n\rangle_{%s}},'
                r'\qquad \langle p,q\rangle_{%s} = %s'
                % (bn, wn, bn, bn, wn, wn, cfg['inner_tex'])))
            lines = r' \\ '.join(
                r'\alpha_{%d} = %s' % (n, sp.latex(alpha[n])) for n in range(d + 1))
            display(Latex(
                r'埃尔米特展开系数（广义傅里叶系数）：$\begin{align*} %s \end{align*}$' % lines))
            display(L(r'\therefore\quad h(x) = ' + latex_basis_form(alpha, conv, approx=False)))
            avec = r' \\ '.join(sp.latex(v) for v in alpha)
            bvec = r' \\ '.join(sp.latex(v) for v in beta)
            display(L(
                r'系数向量：\quad F_{%s}=\begin{bmatrix} %s \end{bmatrix}\quad \text{（}%s_n\text{ 基）},'
                r'\qquad %s,\ F=\begin{bmatrix} %s \end{bmatrix}'
                % (bn, avec, bn, rodrigues_tex(conv), bvec)))
        except Exception as e:
            display(Markdown('<span style="color:#c0392b">输入有误：%s</span>' % e))

btn1 = widgets.Button(description='计算埃尔米特展开', button_style='primary')
btn1.on_click(run_mode1)
demo1a = widgets.Button(description='例题：x²+2x+3')
demo1a.on_click(lambda _: (fill_mode1(['3', '2', '1']), run_mode1(None)))
demo1b = widgets.Button(description='例题：9x⁴-7x³+6x²-11x+3')
demo1b.on_click(lambda _: (fill_mode1(['3', '-11', '6', '-7', '9']), run_mode1(None)))

mode1_ui = widgets.VBox([
    conv_select1,
    widgets.Label('h(x) = a_0 + a_1 x + … + a_d x^d，系数可填整数、小数或分数（如 7/2）'),
    degree_box,
    coeff_box,
    widgets.HBox([btn1, demo1a, demo1b]),
    mode1_out
])

In [4]:
# ===== Cell 5｜模式二：h 未知，输入 m 个点坐标与最高阶 N，运行本 Cell 出界面 =====
conv_select2 = widgets.ToggleButtons(
    options=[('物理学家版 H_n（权 f=e^{-x²}）', 'phys'),
             ('概率学家版 He_n（权 φ=标准正态）', 'prob')],
    value=CONV_NAME, description='约定选择', style={'description_width': 'initial'})

def _on_conv2(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select1' in globals() and conv_select1.value != change['new']:
        conv_select1.value = change['new']

conv_select2.observe(_on_conv2, names='value')

demo_points = ('0.13, 2.417\n0.47, 3.186\n1.26, 4.732\n'
               '1.83, 4.105\n3.14, 2.891\n4.72, 1.634')
points_area = widgets.Textarea(value=demo_points,
                               layout=widgets.Layout(width='340px', height='190px'))
order_box = widgets.BoundedIntText(value=4, min=0, max=20, description='最高阶 N')
order_hint2 = widgets.HTML(
    '<div style="font-size:12.5px;color:#6b7280;line-height:1.6;max-width:760px;margin:2px 0 6px">'
    '当前课堂演示版稳定支持 <b>N≤20</b>；更高阶（例如 1000 阶）会产生超长系数表和多项式，且高阶 Hermite '
    '浮点递推容易溢出或振荡。若以后真要支持 1000 阶，需要分页/虚拟滚动、结果导出和更稳定的数值实现。'
    '数据点较多时，结果输出区与超过 1000 行的残差文本区均可内部滚动。</div>')
mode2_out = widgets.Output(layout=widgets.Layout(
    max_height='580px', overflow='auto', border='1px solid #e5e7eb',
    padding='8px 10px', margin='8px 0'))
prog2_label = widgets.HTML(value='')
prog2 = widgets.IntProgress(value=0, min=0, max=100, description='0%', bar_style='info',
                            style={'bar_color': '#2563eb'},
                            layout=widgets.Layout(width='100%', max_width='760px'))
prog2_box = widgets.VBox([prog2_label, prog2],
                         layout=widgets.Layout(display='none', margin='6px 0', max_width='760px'))

def set_prog2(value, label, bar_style='info'):
    prog2_box.layout.display = ''
    prog2.value = int(max(0, min(100, value)))
    prog2.description = '%d%%' % prog2.value
    prog2.bar_style = bar_style
    prog2_label.value = '<span style="font-size:13px;color:#4b5563">%s</span>' % label

def parse_points(text):
    """解析每行一个点：支持英文逗号、中文逗号或空白分隔，自动按 x 排序"""
    pts = []
    for line in text.strip().splitlines():
        line = line.replace('，', ',').strip()
        if not line:
            continue
        parts = re.split(r'[,\s]+', line)
        if len(parts) != 2:
            raise ValueError('无法解析的行：“%s”，每行需为 x, y 两个数' % line)
        pts.append((float(parts[0]), float(parts[1])))
    if len(pts) < 2:
        raise ValueError('至少需要 2 个点')
    pts.sort(key=lambda t: t[0])
    xv = np.array([p[0] for p in pts])
    if len(set(xv.tolist())) != len(pts):
        raise ValueError('存在重复的 x 值')
    return xv, np.array([p[1] for p in pts])

def run_mode2(_):
    with mode2_out:
        clear_output(wait=True)
        try:
            set_prog2(2, '解析输入点…')
            xv, yv = parse_points(points_area.value)
            N = int(order_box.value)
            m = len(xv)
            if N > 20:
                raise ValueError('当前课堂演示版稳定支持最高 N=20；更高阶会产生超长输出并带来高阶数值不稳定，1000 阶场景需分页/虚拟滚动与导出支持。')
            conv = conv_select2.value
            level, _ = order_warning(m, N)
            banner = warning_banner(m, N)
            if banner:
                display(banner)
            set_prog2(12, '梯形积分计算 αₙ…')
            alpha, Pv = trapezoid_alpha(xv, yv, N, conv)
            set_prog2(38, '计算 Δₙ 稳定性增量…')
            delta_arr = trapezoid_delta(xv, Pv, alpha, conv)
            set_prog2(55, '合并标准多项式与拟合值…')
            c = expand_numeric(alpha, conv)
            yfit = sum(alpha[n] * Pv[n] for n in range(N + 1))
            set_prog2(68, '整理系数展开、SSE 与多项式形式…')
            display(L(static_trap_tex(conv)))
            display(Markdown('数值积分在区间 [%.3g, %.3g] 上近似全空间积分；共 %d 个点、%d 段。'
                             % (xv[0], xv[-1], m, m - 1)))
            display(Markdown(
                r'**停机指标 $\Delta_n$ 的计算原理**。'
                r'老师蓝笔范数定义为 $\Delta_n=\|P_n-P_{n-1}\|_w/\|P_n\|_w$，其中 $P_n=\sum_{j=0}^{n}\alpha_jB_j$、'
                r'$P_n-P_{n-1}=\alpha_nB_n$。写成连续积分形式（理论区间为全实轴，程序在数据区间 $[x_1,x_m]$ 上梯形近似）：'
                r'$$\begin{aligned}'
                r'\Delta_n&=\sqrt{\frac{\int_a^b (P_n-P_{n-1})^2w(x)\,dx}{\int_a^b P_n^2(x)w(x)\,dx}}\\'
                r'&=\sqrt{\frac{\int_a^b [P_n^2(x)-2P_n(x)P_{n-1}(x)+P_{n-1}^2(x)]w(x)\,dx}'
                r'{\int_a^b P_n^2(x)w(x)\,dx}}'
                r'\end{aligned}$$'
                r'第二个等号来自平方差展开，中间的 $-2\int P_n(x)P_{n-1}(x)w(x)\,dx$ 就是老师红笔板书中'
                r'“$P_n$ 与 $P_{n-1}$ 相乘、不平方”的乘积交叉项。再把两个积分在数据节点上梯形离散化即得下面的可计算公式，'
                r'系数表第三列逐行显示其数值（$n\geq1$，$\Delta_0$ 无定义）。'))
            display(L(static_delta_tex(conv)))
            display(Markdown(
                r'**板书红笔“i.e.”原样对照（建议课上与老师核对）**：白板照片演算区中红笔另写有乘积根号形式 '
                r'$\Delta_n=\sqrt{\int_a^b P_n(x)P_{n-1}(x)w(x)\,dx\ \big/\ \int_a^b P_n^2(x)w(x)\,dx}$。'
                r'单独的乘积积分 $\int P_nP_{n-1}w=\langle P_n,P_{n-1}\rangle_w$ 不等于范数分子 '
                r'$\|P_n-P_{n-1}\|_w^2=\int(P_n-P_{n-1})^2w$（差平方展开还含 $P_n^2$、$P_{n-1}^2$ 与系数 $-2$）；'
                r'按默认 6 点数据复算该字面乘积式也得不到白板左侧的 $\Delta_1=0.506050,\Delta_2=0.123712,\ldots$。'
                r'本工具按蓝笔范数定义的节点梯形实现；若老师确认课堂口径就是该乘积根号式，再切换实现即可。'))
            display(Markdown(
                r'每行三列对齐：$\alpha_n$ 为展开系数；$\Delta\alpha_n=\alpha_n-\alpha_{n-1}$ 是系数一阶差分；'
                r'$\Delta_n=\|P_n-P_{n-1}\|_w/\|P_n\|_w$ 是课堂停机准则的加权范数相对增量'
                r'（在数据节点上梯形积分，$n\geq1$），当前最高阶满足 $\Delta_N<0.05$ 时下方给出可停止提示。'))
            def alpha_row(n):
                v = alpha[n]
                if _in_fixed_range(v):
                    head = r'\alpha_{%d} &= %s \approx %s' % (
                        n, fmt_num(v), sp.latex(approx_frac(v)))
                else:
                    head = r'\alpha_{%d} &= %s' % (n, fmt_num(v, latex=True))
                if n >= 1:
                    d = float(v) - float(alpha[n - 1])
                    head += r', & \Delta\alpha_{%d} &= %s' % (
                        n, fmt_num(d, latex=True))
                    dn = delta_arr[n]
                    dn_tex = fmt_num(dn, fixed=6, latex=True) if np.isfinite(dn) else r'\text{--}'
                    head += r', & \Delta_{%d} &= %s' % (n, dn_tex)
                return head
            rows = r' \\ '.join(alpha_row(n) for n in range(N + 1))
            display(Latex(r'$\begin{aligned} %s \end{aligned}$' % rows))
            stop = stop_banner(delta_arr, N, level)
            if stop:
                display(stop)
            sse = float(np.sum((yv - yfit) ** 2))
            mse = sse / m
            display(L(
                r'\begin{aligned} '
                r'\mathrm{SSE} &= \sum_{i=1}^{m}(y_i-F(x_i))^2 = ' + fmt_num(sse, latex=True) + r', \\ '
                r'\mathrm{MSE} &= \frac{1}{m}\sum_{i=1}^{m}(y_i-F(x_i))^2 '
                r'= \frac{\mathrm{SSE}}{m} = ' + fmt_num(mse, latex=True) +
                r' \end{aligned}'))
            display(L(r'F(x) = ' + latex_basis_form(alpha, conv)
                      + r'\quad \text{（埃尔米特形式，分数为近似值）}'))
            display(L(r'F(x) = ' + latex_standard_poly(c)
                      + r'\quad \text{（标准多项式形式，分数为近似值）}'))
            set_prog2(76, '生成残差表…')
            if m <= 1000:
                tbl_lines = ['| i | x_i | y_i | F(x_i) | 残差 |',
                             '|---|---|---|---|---|']
                for i in range(m):
                    tbl_lines.append('| %d | %.4g | %.4g | %.4g | %.4g |' % (
                        i + 1, xv[i], yv[i], yfit[i], yv[i] - yfit[i]))
                    if (i + 1) % 2000 == 0 or i == m - 1:
                        set_prog2(76 + 16 * (i + 1) / m, '生成残差表 %d/%d' % (i + 1, m))
                display(Markdown('\n'.join(tbl_lines)))
            else:
                display(Markdown('**残差表（%d 行）**：大点数改用框内滚动文本区，制表符分隔，可全选复制到 Excel。' % m))
                tsv_lines = ['i\tx_i\ty_i\tF(x_i)\tresidual']
                for i in range(m):
                    tsv_lines.append('%d\t%.6g\t%.6g\t%.6g\t%.6g' % (
                        i + 1, xv[i], yv[i], yfit[i], yv[i] - yfit[i]))
                    if (i + 1) % 2000 == 0 or i == m - 1:
                        set_prog2(76 + 16 * (i + 1) / m, '生成滚动残差表 %d/%d' % (i + 1, m))
                display(widgets.Textarea(
                    value='\n'.join(tsv_lines), disabled=True,
                    layout=widgets.Layout(height='380px', width='100%', max_width='760px')))
            set_prog2(94, '绘制拟合图…')
            grid = np.linspace(xv[0], xv[-1], 400)
            Pg = basis_values(grid, N, conv)
            yg = sum(alpha[n] * Pg[n] for n in range(N + 1))
            plt.figure(figsize=(7, 3.4))
            plt.plot(grid, yg, label='F(x)')
            plt.scatter(xv, yv, color='crimson', zorder=3, label='data points')
            plt.xlabel('x'); plt.ylabel('y'); plt.legend()
            plt.grid(alpha=.3); plt.tight_layout()
            plt.show()
            set_prog2(100, '完成：%d 个点、最高阶 N=%d' % (m, N), 'success')
        except Exception as e:
            set_prog2(prog2.value, '计算中断：%s' % e, 'danger')
            display(Markdown('<span style="color:#c0392b">输入有误：%s</span>' % e))

# ----- 模式二随机点生成器（只填入数据框，不自动计算）-----
gen_count = widgets.Text(value='', placeholder='必填，如 20', description='点数',
                         style={'description_width': 'initial'})
gen_range_on = widgets.Checkbox(
    value=False, description='限定坐标范围（勾选后 x、y 共用同一均匀区间）',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='auto'))
gen_min = widgets.FloatText(value=-2.0, description='最小值',
                            style={'description_width': 'initial'})
gen_max = widgets.FloatText(value=3.0, description='最大值',
                            style={'description_width': 'initial'})
gen_order = widgets.Dropdown(options=[('不排序（随机顺序）', 'none'),
                                      ('按 x 递增', 'asc'),
                                      ('按 x 递减', 'desc')],
                             value='asc', description='顺序',
                             style={'description_width': 'initial'})
gen_btn = widgets.Button(description='生成随机点', button_style='info')
gen_msg = widgets.HTML(value='')
gen_hint = widgets.HTML(
    '<div style="font-size:12.5px;color:#6b7280;line-height:1.6;max-width:690px">'
    '点数范围 2–20000。不勾范围：x、y 均按标准正态 N(0,1) 生成（无硬边界）；勾选范围：x、y 均在 '
    '[最小值, 最大值] 内均匀随机，且最小值必须严格小于最大值。直接修改最小/最大值会自动勾选范围；'
    '递减只改变填入文本框的顺序，点击计算后 parse_points 仍会自动按 x 升序排序；生成后需再点“计算完整多项式”。</div>')

def _toggle_gen_range(change):
    if not gen_range_on.value:
        gen_msg.value = ''

def _auto_enable_gen_range(change):
    if not gen_range_on.value:
        gen_range_on.value = True

gen_range_on.observe(_toggle_gen_range, names='value')
gen_min.observe(_auto_enable_gen_range, names='value')
gen_max.observe(_auto_enable_gen_range, names='value')

def _set_gen_msg(text, ok=False):
    color = '#166534' if ok else '#c0392b'
    gen_msg.value = '<div style="color:%s;font-size:13px;margin:4px 0">%s</div>' % (color, text)

def generate_random_points(_):
    raw_count = (gen_count.value or '').strip()
    if raw_count == '':
        _set_gen_msg('请先输入点数量，再点生成。')
        return
    try:
        m = int(raw_count)
    except Exception:
        _set_gen_msg('点数量必须是整数。')
        return
    if m < 2:
        _set_gen_msg('点数量必须是 ≥2 的整数（至少 2 个点才能进入模式二）。')
        return
    if m > 20000:
        _set_gen_msg('单次最多生成 20000 个点；更多点建议分批生成或直接粘贴数据。')
        return
    rng = np.random.default_rng()
    if gen_range_on.value:
        lo, hi = float(gen_min.value), float(gen_max.value)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            _set_gen_msg('最小值和最大值都必须是有效数字。')
            return
        if lo == hi:
            _set_gen_msg('最小值与最大值相等，已禁止生成；请让最小值严格小于最大值。')
            return
        if lo > hi:
            _set_gen_msg('左侧最小值大于右侧最大值（左大右小），已禁止生成；请改成左小右大。')
            return
        xs = rng.uniform(lo, hi, size=m)
        ys = rng.uniform(lo, hi, size=m)
        range_text = 'x、y 均匀随机于 [%.6g, %.6g]' % (lo, hi)
    else:
        xs = rng.standard_normal(size=m)
        ys = rng.standard_normal(size=m)
        range_text = 'x、y 按标准正态 N(0,1) 随机（未限定范围）'
    order = gen_order.value
    pts = list(zip(xs.tolist(), ys.tolist()))
    if order == 'asc':
        pts.sort(key=lambda p: p[0])
    elif order == 'desc':
        pts.sort(key=lambda p: p[0], reverse=True)
    points_area.value = '\n'.join('%.6f, %.6f' % (x, y) for x, y in pts)
    order_text = {'asc': '按 x 递增填入',
                  'desc': '按 x 递减填入（计算时仍自动升序）',
                  'none': '保持随机顺序'}[order]
    _set_gen_msg('已生成 %d 个点：%s；%s。确认后点“计算完整多项式”。'
                 % (m, range_text, order_text), ok=True)

gen_btn.on_click(generate_random_points)
gen_ui = widgets.VBox([
    widgets.HTML('<b>随机生成测试点（只填入上方数据框，不自动计算）</b>'),
    widgets.HBox([gen_count, gen_btn]),
    gen_range_on,
    widgets.HBox([gen_min, gen_max]),
    gen_order,
    gen_hint,
    gen_msg
], layout=widgets.Layout(border='1px dashed #94a3b8', padding='10px 12px',
                         margin='8px 0', max_width='760px'))

btn2 = widgets.Button(description='计算完整多项式', button_style='primary')
btn2.on_click(run_mode2)

mode2_ui = widgets.VBox([
    conv_select2,
    widgets.Label('每行输入一个点，格式 x, y（支持空格或中文逗号分隔）'),
    points_area,
    gen_ui,
    order_box,
    order_hint2,
    prog2_box,
    btn2,
    mode2_out
])

tab = widgets.Tab(children=[mode1_ui, mode2_ui])
tab.titles = ('模式一：已知 h 多项式', '模式二：已知数据点')
display(tab)

In [5]:
# ===== Cell 6｜离线自检：两约定对照例题，可随时重跑 =====
# --- 物理学家版：h=x^2+2x+3 -> F(7/2,-1,1/4) ---
alpha1, beta1, h1 = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'phys')
assert beta1 == [sp.Rational(7, 2), sp.Rational(-1), sp.Rational(1, 4)]
assert alpha1 == [sp.Rational(7, 2), sp.Rational(1), sp.Rational(1, 4)]

# --- 概率学家版：同一函数 -> F_He(4,2,1)，Rodrigues 基 (4,-2,1) ---
alpha1p, beta1p, _ = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'prob')
assert alpha1p == [sp.Rational(4), sp.Rational(2), sp.Rational(1)]
assert beta1p == [sp.Rational(4), sp.Rational(-2), sp.Rational(1)]

# --- 9x^4 例题：两约定均可无损还原 ---
coeffs4 = [sp.Rational(3), sp.Rational(-11), sp.Rational(6),
           sp.Rational(-7), sp.Rational(9)]
for conv in ('phys', 'prob'):
    a2, _, h2 = exact_expansion(coeffs4, conv)
    rebuilt = sp.expand(sum(a2[n] * exact_basis_poly(n, conv) for n in range(5)))
    assert rebuilt == h2

# --- 稠密点梯形法：两约定均逼近精确系数（概率学家版尾部更重，网格取 ±5）---
xg = np.linspace(-5, 5, 501)
yg = xg ** 2 + 2 * xg + 3
al_phys, _ = trapezoid_alpha(xg, yg, 2, 'phys')
err_phys = float(np.abs(al_phys - np.array([3.5, 1.0, 0.25])).max())
assert err_phys < 1e-3
al_prob, _ = trapezoid_alpha(xg, yg, 2, 'prob')
err_prob = float(np.abs(al_prob - np.array([4.0, 2.0, 1.0])).max())
assert err_prob < 1e-3

# --- 三级阶数提示逻辑 ---
assert order_warning(6, 4)[0] == 'caution'     # 6 点估计 5 参数
assert order_warning(6, 9)[0] == 'danger'      # 参数多于点数，欠定
assert order_warning(101, 20)[0] == 'ok'       # 稠密点，无提示
assert warning_banner(101, 20) == ''

# --- Δα 一阶差分与 SSE 校验（稠密网格，物理学家版）---
d_phys = np.diff(al_phys)
assert np.allclose([d_phys[n - 1] for n in range(1, len(al_phys))],
                   [al_phys[n] - al_phys[n - 1] for n in range(1, len(al_phys))])
assert np.isclose(d_phys.sum(), al_phys[-1] - al_phys[0])   # 望远镜求和
P_g = basis_values(xg, 2, 'phys')
yfit_g = sum(al_phys[n] * P_g[n] for n in range(3))
sse_g = float(np.sum((yg - yfit_g) ** 2))
assert sse_g >= 0

# --- 课堂 Truncation Rule：节点梯形口径 Δ_n 复现白板 6 点数值（2026-11-19 课）---
x6 = np.array([0.13, 0.47, 1.26, 1.83, 3.14, 4.72])
y6 = np.array([2.417, 3.186, 4.732, 4.105, 2.891, 1.634])
a6, Pv6 = trapezoid_alpha(x6, y6, 5, 'phys')
d6 = trapezoid_delta(x6, Pv6, a6, 'phys')
for _n, _v in {1: 0.506050, 2: 0.123712, 3: 0.092488,
               4: 0.042527, 5: 0.021085}.items():
    assert abs(d6[_n] - _v) < 5e-6, (_n, d6[_n])
assert d6[3] >= 0.05 and d6[4] < 0.05      # 阈值 0.05 时首次在 n=4 可停止
assert stop_banner(d6, 4, 'caution') != ''  # 黄级数据下仍渲染停机提示
assert stop_banner(d6, 5, 'danger') != ''   # 红级欠定时降级提示也正常渲染
assert stop_banner(d6, 99, 'ok') == ''      # 越界 N 安全返回空
assert r'\Delta_n' in static_delta_tex('phys')
assert r'He_n' in static_delta_tex('prob')  # 概率版公式正确替换基符号

print('自检通过：')
print('  1. 物理学家版 h=x^2+2x+3 -> F(7/2, -1, 1/4)，与笔记图 2 完全一致')
print('  2. 概率学家版 h=x^2+2x+3 -> F_He(4, 2, 1)，Rodrigues 基 (4, -2, 1)')
print('  3. 9x^4 例题两约定均可无损还原原函数')
print('  4. 稠密点梯形法最大偏差：物理学家版 %.2e，概率学家版 %.2e（均 < 1e-3）'
      % (err_phys, err_prob))
print('  5. 三级阶数提示（黄/红/无）分级逻辑正确，高阶不再硬拦截')
print('  6. Δα 一阶差分望远镜求和成立，SSE 计算正常（稠密网格 SSE=%.2e）' % sse_g)
print('  7. 课堂 Δ_n（节点梯形范数）复现白板数值，0.05 停机提示正常')
display(Latex(
    r'h(x)=x^2+2x+3=\frac{7}{2}H_0+H_1+\frac{1}{4}H_2'
    r'=4He_0+2He_1+He_2'))

自检通过：
  1. 物理学家版 h=x^2+2x+3 -> F(7/2, -1, 1/4)，与笔记图 2 完全一致
  2. 概率学家版 h=x^2+2x+3 -> F_He(4, 2, 1)，Rodrigues 基 (4, -2, 1)
  3. 9x^4 例题两约定均可无损还原原函数
  4. 稠密点梯形法最大偏差：物理学家版 6.88e-11，概率学家版 2.98e-05（均 < 1e-3）
  5. 三级阶数提示（黄/红/无）分级逻辑正确，高阶不再硬拦截
  6. Δα 一阶差分望远镜求和成立，SSE 计算正常（稠密网格 SSE=4.56e-15）
  7. 课堂 Δ_n（节点梯形范数）复现白板数值，0.05 停机提示正常


<IPython.core.display.Latex object>